In [0]:
%sql
drop table raw.customers

In [0]:
%sql
show partitions raw.customers;



In [0]:
%sql
select *  from processed.customers where customer_id='PW-19240'

In [0]:
spark.conf.get("spark.sql.sources.partitionOverwriteMode")

In [0]:
import sys
import os
test_path = os.path.abspath('../../tests')
if test_path not in sys.path:
    sys.path.append(test_path)
sys.path

src_path=os.path.abspath('../../src')
if src_path not in sys.path:
    sys.path.append(src_path)
sys.path
import pytest
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, to_date # Added current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DateType, ArrayType, BooleanType, TimestampType
from datetime import date, datetime, timedelta # Import timedelta for date calculations
from unittest.mock import MagicMock, patch
# Assuming your ingestion code is in a module named 'ecomm_ingestion_pipeline'
# You might need to adjust this import based on your project structure
from transform_functions.ingest_customer_data_functions import (
    customer_column_standardize,
    handle_customer_id_nulls,
    clean_customer_data,
    perform_customer_data_quality_checks,
    load_customer_data,
    ingest_customer_pipeline,
    customer_schema, # Import the raw schema
    processed_customer_schema # Import the processed schema
)
from common.functions import add_ingestion_date


def test_load_customer_data_scd_type_2(spark_session: spark, temp_delta_path: str):
    """
    Tests the SCD Type 2 implementation in load_customer_data for the 'processed' schema,
    using a temporary file path for isolation.
    Scenario:
    1. Initial load of customer data.
    2. A new batch arrives with:
        - An update to an existing customer's address (C001).
        - A new customer (C004).
        - An existing customer with no changes (C002).
        - A customer that was in the first batch but is not in the second batch (C003).
    3. Verify the state of the Delta table after the merge.
    """
    print("Starting SCD Type 2 test for load_customer_data.")
    target_delta_path = temp_delta_path

    # Create the database if it doesn't exist (needed for table creation)
    spark_session.sql(f"CREATE DATABASE IF NOT EXISTS processed LOCATION '{target_delta_path}'")

    # Define the schema for the input DF to load_customer_data (df_final_for_processed)
    # This schema should have 13 fields (11 original + file_date + dq_issues)
    # 'ingestion_timestamp' is added by add_ingestion_date *before* this function is called.
    input_schema_for_load_customer = StructType([
        StructField("customer_id", StringType(), False),
        StructField("customer_name", StringType(), True),
        StructField("email", StringType(), True),
        StructField("phone", StringType(), True),
        StructField("address", StringType(), True),
        StructField("segment", StringType(), True),
        StructField("country", StringType(), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("postal_code", StringType(), True),
        StructField("region", StringType(), True),
        StructField("file_date", DateType(), False),
        StructField("dq_issues", ArrayType(StringType()), True),
        # ingestion_timestamp is added by add_ingestion_date function, not part of this input schema
    ])

    # --- Initial Load (Day 1) ---
    file_date_day1 = "2025-01-01"
    initial_data = [
        ("C001", "Alice Smith", "alice@example.com", "123-456-7890", "123 Old St", "Consumer", "USA", "New York", "NY", "10001", "East", date(2025,1,1), []),
        ("C002", "Bob Johnson", "bob@example.com", "987-654-3210", "456 Main St", "Corporate", "USA", "Los Angeles", "CA", "90210", "West", date(2025,1,1), []),
        ("C003", "Charlie Brown", "charlie@example.com", "555-111-2222", "789 Pine Ln", "Home Office", "Canada", "Toronto", "ON", "M1A 1A1", "Central", date(2025,1,1), []),
    ]
    df_initial_load = spark_session.createDataFrame(initial_data, input_schema_for_load_customer)
    df_initial_load_with_ingestion = add_ingestion_date(df_initial_load) # Add ingestion_timestamp

    print(f"Performing initial load for {target_delta_path} on {file_date_day1}")
    # Pass the temporary path as the 'schemaname' for testing purposes
    load_customer_data(spark_session, "processed", df_initial_load_with_ingestion, file_date_day1) # Use "processed" as schemaname

    # Verify initial load state
    current_table_df = spark_session.read.format("delta").table("processed.customers") # Read from table name
    current_table_df.show(truncate=False)
    assert current_table_df.count() == 3
    for row in current_table_df.select("customer_id", "effective_start_date", "effective_end_date", "is_current", "last_updated_timestamp", "ingestion_timestamp").collect():
        assert row["is_current"] is True
        assert row["effective_start_date"] == date(2025, 1, 1)
        assert row["effective_end_date"] is None
        assert row["last_updated_timestamp"] is not None
        assert isinstance(row["last_updated_timestamp"], datetime)
        assert row["ingestion_timestamp"] is not None # Check ingestion_timestamp
        assert isinstance(row["ingestion_timestamp"], datetime)
        # Check if timestamp is roughly around the load time (within a reasonable window)
        assert datetime.now() - timedelta(minutes=1) <= row["last_updated_timestamp"] <= datetime.now() + timedelta(minutes=1)
        assert datetime.now() - timedelta(minutes=1) <= row["ingestion_timestamp"] <= datetime.now() + timedelta(minutes=1)
    print("Initial load verified.")
    
    # --- Second Load (Day 2) with Changes ---
    file_date_day2 = "2025-01-02"
    second_batch_data = [
        # C001: Address change
        ("C001", "Alice Smith", "alice@example.com", "123-456-7890", "123 New Ave", "Consumer", "USA", "New York", "NY", "10001", "East", date(2025,1,2), []),
        # C002: No changes
        ("C002", "Bob Johnson", "bob@example.com", "987-654-3210", "456 Main St", "Corporate", "USA", "Los Angeles", "CA", "90210", "West", date(2025,1,2), []),
        # C004: New customer
        ("C004", "Diana Prince", "diana@example.com", "111-222-3333", "Wonder Ave", "Consumer", "USA", "Themyscira", "AM", "99999", "South", date(2025,1,2), []),
    ]
    df_second_batch = spark_session.createDataFrame(second_batch_data, input_schema_for_load_customer)
    df_second_batch_with_ingestion = add_ingestion_date(df_second_batch) # Add ingestion_timestamp

    print(f"Performing second load for {target_delta_path} on {file_date_day2}")
    load_customer_data(spark_session, "processed", df_second_batch_with_ingestion, file_date_day2) # Use "processed" as schemaname

    # Verify state after second load
    final_table_df = spark_session.read.format("delta").table("processed.customers") # Read from table name
    final_table_df.show(truncate=False) # For debugging

    # C001: Should have two records
    c001_records = final_table_df.filter(col("customer_id") == "C001").orderBy("effective_start_date").collect()
    assert len(c001_records) == 2

    # Old C001 record (closed out)
    old_c001 = c001_records[0]
    assert old_c001.address == "123 Old St"
    assert old_c001.is_current is False
    assert old_c001.effective_start_date == date(2025, 1, 1)
    assert old_c001.effective_end_date == date(2025, 1, 1) # (file_date_day2 - 1 day)
    assert old_c001.last_updated_timestamp is not None # Should be updated
    assert isinstance(old_c001.last_updated_timestamp, datetime)
    assert old_c001.ingestion_timestamp is not None # Check ingestion_timestamp
    assert isinstance(old_c001.ingestion_timestamp, datetime)
    assert datetime.now() - timedelta(minutes=1) <= old_c001.last_updated_timestamp <= datetime.now() + timedelta(minutes=1)


    # New C001 record
    new_c001 = c001_records[1]
    assert new_c001.address == "123 New Ave"
    assert new_c001.is_current is True
    assert new_c001.effective_start_date == date(2025, 1, 2)
    assert new_c001.effective_end_date is None
    assert new_c001.last_updated_timestamp is not None # Should be populated
    assert isinstance(new_c001.last_updated_timestamp, datetime)
    assert new_c001.ingestion_timestamp is not None # Check ingestion_timestamp
    assert isinstance(new_c001.ingestion_timestamp, datetime)
    assert datetime.now() - timedelta(minutes=1) <= new_c001.last_updated_timestamp <= datetime.now() + timedelta(minutes=1)


    # C002: Should have one record (unchanged)
    c002_records = final_table_df.filter(col("customer_id") == "C002").collect()
    assert len(c002_records) == 1
    assert c002_records[0].is_current is True
    assert c002_records[0].effective_start_date == date(2025, 1, 1)
    assert c002_records[0].effective_end_date is None
    assert c002_records[0].last_updated_timestamp is not None
    assert c002_records[0].ingestion_timestamp is not None # Check ingestion_timestamp
    # For unchanged records, last_updated_timestamp should NOT change. It should be the initial load timestamp.
    # This requires capturing the initial timestamp for comparison.
    # We can't directly check against `datetime.now()` for unchanged records.
    # Instead, we should check if it's within the initial load time window.
    # For simplicity in this test, we'll just check it's not null and is a datetime.
    assert isinstance(c002_records[0].last_updated_timestamp, datetime)
    assert isinstance(c002_records[0].ingestion_timestamp, datetime)


    # C003: Should still exist but be marked as current (not in second batch implies no new activity)
    # The current merge logic only updates when matched AND changed.
    # It does not explicitly expire records that are *not* in the incoming batch.
    # This is a common SCD Type 2 behavior. If you want to expire un-seen records, you'd need a different merge strategy (e.g., a "full outer join" approach).
    # Based on the current code, C003 should remain `is_current = true` with `effective_end_date = None`.
    c003_records = final_table_df.filter(col("customer_id") == "C003").collect()
    assert len(c003_records) == 1
    assert c003_records[0].is_current is True
    assert c003_records[0].effective_start_date == date(2025, 1, 1)
    assert c003_records[0].effective_end_date is None
    assert c003_records[0].last_updated_timestamp is not None
    assert c003_records[0].ingestion_timestamp is not None # Check ingestion_timestamp
    assert isinstance(c003_records[0].last_updated_timestamp, datetime)
    assert isinstance(c003_records[0].ingestion_timestamp, datetime)


    # C004: Should be a new record
    c004_records = final_table_df.filter(col("customer_id") == "C004").collect()
    assert len(c004_records) == 1
    assert c004_records[0].is_current is True
    assert c004_records[0].effective_start_date == date(2025, 1, 2)
    assert c004_records[0].effective_end_date is None
    assert c004_records[0].last_updated_timestamp is not None
    assert c004_records[0].ingestion_timestamp is not None # Check ingestion_timestamp
    assert isinstance(c004_records[0].last_updated_timestamp, datetime)
    assert isinstance(c004_records[0].ingestion_timestamp, datetime)
    assert datetime.now() - timedelta(minutes=1) <= c004_records[0].last_updated_timestamp <= datetime.now() + timedelta(minutes=1)

    print("SCD Type 2 merge verified.")
    print("SCD Type 2 test for load_customer_data PASSED.")


In [0]:
dbutils.fs.rm("/tmp", recurse=True)

In [0]:
test_load_customer_data_scd_type_2(spark,"/tmp")